# IMD2020 evaluation

Evaluation of the spatial-frequency fusion model for image-level forgery detection and pixel localization.
The external test manifest contains 2,424 images: 414 authentic and 2,010 forged, grouped by source original.
The notebook audits inputs, loads the trained checkpoint, calculates metrics and exports PNG figures.

Final decision rule: predicted image label 0 gives an empty mask; label 1 permits pixels above the configured pixel threshold.
Existing IMD2020 measurements were recorded before this gate was added. See results/README.md for the distinction
between those historical measurements and the gated presentation results.

Data layout: IMD2020/<source_id>/<source_id>_orig.jpg, with edited images and their matching *_mask.png files in the same folder.
No training or threshold selection takes place in this notebook.


## 1. Environment and configuration

Install requirements.txt in a Python environment with a suitable PyTorch build. Start Jupyter from the project folder.
The original training statistics and split audit are read from artifacts/ and checked against the checkpoint identity.
Choose a model, device, test manifest and frozen thresholds below. Dataset paths are relative to the project root.
The default evaluation uses native-resolution clean images. Additional conditions must be configured before inference.


In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, time, random, importlib.metadata
from datetime import datetime, timezone

# Set an explicit project directory only when the kernel starts outside this project.
PROJECT_OVERRIDE = None
candidates = ([Path(PROJECT_OVERRIDE).expanduser()] if PROJECT_OVERRIDE else
              [Path.cwd(), *Path.cwd().parents])
PROJECT = next((p.resolve() for p in candidates if (p / 'models/model.py').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Open Jupyter from the project folder or set PROJECT_OVERRIDE to its location in this kernel.')
sys.path.insert(0, str(PROJECT))

DATA_ROOT = PROJECT / 'datasets/imd2020'
STATS_PATH = PROJECT / 'artifacts/dct_stats.json'
AUDIT_PATH = PROJECT / 'artifacts/casia_clean_split.json'
MANIFEST = PROJECT / 'evaluation/manifests/imd2020_test.csv'
# Use imd2020_preview.csv for a small workflow check, then restore the full manifest.
MODEL_SPECS = {
    'Fusion finetuned': ('fusion', '04_fusion_finetuned'),
    # 'Spatial baseline': ('spatial_only', '01_spatial_baseline'),
    # 'Frequency baseline': ('frequency_only', '02_frequency_baseline'),
    # 'Fusion warmup': ('fusion', '03_fusion_warmup'),
}
SEED = 42
IMAGE_THRESHOLD = 0.5
PIXEL_THRESHOLD = 0.5
MASK_FOREGROUND = 'white'  # Verify visually below; switch to black only if the annotation convention requires it.
CONDITIONS = ['clean']
# Optional robustness study: ['clean', 'jpeg70', 'scale05', 'blur1', 'noise5']
# All conditions start from original input pixels; masks stay on the native pixel grid.
BOOTSTRAPS = 1000
EXAMPLES_PER_CATEGORY = 2
SAVE_ALL_PREDICTIONS = True  # compressed float32 probabilities + uint8 binary PNGs; can use substantial disk space
DEVICE_REQUEST = 'auto'     # 'auto', 'cpu', or 'cuda'
RUN = PROJECT / 'results/imd2020' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN.mkdir(parents=True, exist_ok=False)
(RUN / 'figures').mkdir()
(RUN / 'predictions').mkdir()
print('Project:', PROJECT, '\nOutput:', RUN)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch
from PIL import Image
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, confusion_matrix, brier_score_loss)
from evaluation.prepare_imd2020 import prepare
from evaluation.evaluate import ManifestDataset, model_sample, predict, training_assets, digest
from evaluation.metrics import EvaluationMetrics, detection_metrics
from evaluation.degradations import CONDITIONS as AVAILABLE_CONDITIONS
from models.model import ForgeryModel

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE_REQUEST == 'auto' else DEVICE_REQUEST)
assert 0 <= IMAGE_THRESHOLD <= 1 and 0 < PIXEL_THRESHOLD <= 1
assert MASK_FOREGROUND in ('white', 'black')
assert CONDITIONS and len(CONDITIONS) == len(set(CONDITIONS)) and 'clean' in CONDITIONS
assert set(CONDITIONS) <= set(AVAILABLE_CONDITIONS)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 13,
    'axes.titlesize': 16, 'axes.labelsize': 13, 'axes.spines.top': False,
    'axes.spines.right': False, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'svg.fonttype': 'none', 'figure.facecolor': 'white'})
COLORS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']
figure_index = []
def export(fig, name, caption):
    fig.savefig(RUN / 'figures' / f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    figure_index.append({'figure': name, 'caption': caption})
    plt.show()
    plt.close(fig)

print('Device:', DEVICE)
print(prepare(DATA_ROOT, MANIFEST.parent, seed=SEED))
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
manifest.to_csv(RUN / 'test_manifest.csv', index=False)
dataset = ManifestDataset(DATA_ROOT, MANIFEST, MASK_FOREGROUND)


## 2. Audit the test data before inference

Pairing uses exact stems, never file order. Unpaired images, orphan masks, duplicate IDs, different image/mask
dimensions, empty forged masks, and invalid label/mask combinations stop the run. No invalid sample is silently dropped.
Inspect the mask convention: **white means forged**, black means authentic background in the normalized masks.
The original source image can have a different size/content from an edited image; it is contextual reference only.
Metrics compare the edited input with its own aligned ground-truth mask.


In [ ]:
audit_rows = []
for i in tqdm(range(len(dataset)), desc='Validate images and masks'):
    raw = dataset[i]  # Opens and validates dimensions, binary mask, and label agreement.
    h, w = raw['mask'].shape[1:]
    row = manifest.iloc[i]
    audit_rows.append({'id': row.id, 'group': row.group, 'label': int(raw['label']),
        'height': h, 'width': w, 'megapixels': h*w/1e6,
        'foreground_fraction': float(raw['mask'].mean()),
        'image_sha256': digest(DATA_ROOT / row.image)})
audit = pd.DataFrame(audit_rows)
audit.to_csv(RUN / 'dataset_audit.csv', index=False)
duplicates = audit[audit.duplicated('image_sha256', keep=False)]
duplicates.to_csv(RUN / 'duplicate_images.csv', index=False)
if not duplicates.empty:
    raise ValueError('Exact duplicate image files found. Review duplicate_images.csv before evaluation to avoid duplicate weighting.')
display(audit.groupby('label').agg(images=('id','size'), source_groups=('group','nunique'),
    median_megapixels=('megapixels','median')))
display(audit.describe(numeric_only=True))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout='constrained')
counts = audit.label.value_counts().reindex([0,1], fill_value=0)
axes[0].bar(['Authentic (0)', 'Forged (1)'], counts, color=COLORS[:2])
axes[0].set(title='Test-set class balance', ylabel='Images')
axes[1].hist(audit.megapixels, bins=25, color=COLORS[0])
axes[1].set(title='Native image sizes', xlabel='Megapixels', ylabel='Images')
axes[2].hist(100*audit.loc[audit.label.eq(1), 'foreground_fraction'], bins=25, color=COLORS[1])
axes[2].set(title='Forged region sizes', xlabel='Ground-truth forged area (%)', ylabel='Forged images')
export(fig, '01_dataset_profile', 'Dataset composition and native-resolution foreground sizes; these are data properties, not model scores.')

for i in manifest.index[manifest.label.eq(1)][:3]:
    raw = dataset[int(i)]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.8), layout='constrained')
    ax[0].imshow(np.moveaxis(raw['rgb'], 0, -1)); ax[0].set_title(f'Forged input: {raw["path"]}')
    with Image.open(DATA_ROOT / manifest.iloc[i]['mask']) as im:
        ax[1].imshow(np.asarray(im), cmap='gray'); ax[1].set_title('Mask file as stored')
    ax[2].imshow(raw['mask'][0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax[2].set_title('Normalized GT: white = forged')
    for a in ax: a.axis('off')
    export(fig, f'02_mask_audit_{i}', 'Verify mask alignment and foreground polarity before interpreting localization scores.')


## 3. How this model works

The spatial branch uses an HRNet-W18 encoder to extract visual features. The frequency branch uses
64 zigzag-ordered signed-log block-DCT channels normalized by training statistics and three SRM residual maps.
Fusion learns weights between branches at four scales. An FPN decoder creates pixel logits; sigmoid gives a
forgery-probability map. A separate pooled-feature classifier produces the image-level probability.
The final image decision gates localization: label 0 gives an empty mask; label 1 permits the thresholded pixel output.
Overlays below visualize predicted probabilities; they are not causal explanations or attention maps.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5), layout='constrained')
ax.set(xlim=(0, 16), ylim=(0, 5)); ax.axis('off')
def box(x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x,y), w,h, boxstyle='round,pad=0.1', facecolor=color, alpha=.16, edgecolor=color))
    ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=12)
def arrow(a,b): ax.annotate('', xy=b, xytext=a, arrowprops={'arrowstyle':'->','lw':2,'color':'#444444'})
box(.2,2,2,1,'Input RGB\nNative resolution',COLORS[0])
box(3,3.2,3,1,'Spatial branch\nHRNet-W18',COLORS[0])
box(3,.7,3,1,'DCT + SRM\nFrequency encoder',COLORS[1])
box(7,2,2.3,1,'Multiscale\ngated fusion',COLORS[2])
box(10,2,2,1,'FPN decoder',COLORS[2])
box(13,3.2,2.6,1,'Pixel head\nProbability mask',COLORS[3])
box(13,.7,2.6,1,'Image head\nP(forged)',COLORS[3])
for a,b in [((2.2,2.5),(3,3.7)),((2.2,2.5),(3,1.2)),((6,3.7),(7,2.5)),((6,1.2),(7,2.5)),
            ((9.3,2.5),(10,2.5)),((12,2.5),(13,3.7)),((12,2.5),(13,1.2))]: arrow(a,b)
export(fig, '03_model_workflow', 'Fusion architecture implemented in this repository. Standalone modes retain only the selected encoder branch.')


## 4. Load exact training assets and evaluate

Thresholds are frozen at the configured values (0.5 by default); replace them only with thresholds selected
using an independent validation set. Test curves later describe sensitivity and must not be used to choose
reported operating points. Inference uses float32, one image at a time, no resizing, and padding to a multiple
of 32. Padded pixels are excluded. Large images can exceed GPU memory; use CPU if necessary rather than
silently resizing the test set. Runtime includes preprocessing, inference, and metric calculation, excluding disk exports.

Localization IoU/Dice are macro averages over forged images only. Authentic masks have undefined foreground
IoU/Dice and instead contribute false-positive area. Pixel accuracy can be misleading because background dominates.
Pixel ROC-AUC is computed per image then averaged over eligible forged images; it is not image-level ROC-AUC.


In [ ]:
for p in [STATS_PATH, AUDIT_PATH]:
    if not p.is_file():
        raise FileNotFoundError(f'Missing original training asset: {p}. Retrieve it from training output; do not fit on IMD2020.')
stats, data_id, training_audit = training_assets(STATS_PATH, AUDIT_PATH)
models_metadata, frames, summaries = {}, [], []
prediction_dirs = {}
groups_by_id = manifest.set_index('id')['group'].to_dict()
for number, (name, (mode, folder)) in enumerate(MODEL_SPECS.items()):
    ckpt = PROJECT / 'results/checkpoints' / folder / 'best.pt'
    ckpt_hash = digest(ckpt)
    state = torch.load(ckpt, map_location='cpu', weights_only=True)
    if state.get('version') != 1 or state.get('mode') != mode or state.get('data_id') != data_id:
        raise ValueError(f'Checkpoint format, mode, or training-asset identity mismatch: {ckpt}')
    if state.get('config', {}).get('smoke', False):
        raise ValueError('Synthetic smoke-test weights cannot be reported as trained results.')
    model = ForgeryModel(mode, pretrained_spatial=False)
    model.load_state_dict(state['model'], strict=True)
    models_metadata[name] = {'path': str(ckpt), 'sha256': ckpt_hash, 'mode': mode,
        'stage': state['stage'], 'epoch': state['epoch'], 'data_id': data_id}
    del state
    if digest(ckpt) != ckpt_hash:
        raise RuntimeError('Checkpoint changed during loading.')
    model.to(DEVICE).eval()
    for condition in CONDITIONS:
        destination = RUN / 'predictions' / f'model_{number:02d}' / condition
        destination.mkdir(parents=True)
        prediction_dirs[(name, condition)] = destination
        meter = EvaluationMetrics(PIXEL_THRESHOLD, IMAGE_THRESHOLD)
        times = []
        for i in tqdm(range(len(dataset)), desc=f'{name} / {condition}'):
            raw = dataset[i]
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            start = time.perf_counter()
            sample = model_sample(raw, mode, condition, stats, SEED)
            raw_probability, score = predict(model, sample, DEVICE)
            probability = raw_probability if score >= IMAGE_THRESHOLD else np.zeros_like(raw_probability)
            result = meter.update(raw['path'], raw['label'], score, sample['mask'][0], probability, sample['valid'][0])
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            times.append(time.perf_counter() - start)
            h, w = raw['mask'].shape[1:]
            if SAVE_ALL_PREDICTIONS:
                np.savez_compressed(destination / f'{i:06d}.npz', probability=probability[:h,:w].astype(np.float32), raw_probability=raw_probability[:h,:w].astype(np.float32))
                Image.fromarray((probability[:h,:w] >= PIXEL_THRESHOLD).astype(np.uint8)*255).save(destination / f'{i:06d}_mask.png')
        table = pd.DataFrame(meter.rows)
        table['model'], table['condition'] = name, condition
        table['group'] = table.sample_id.map(groups_by_id)
        table['seconds'] = times
        table['prediction_file'] = [str((destination / f'{i:06d}.npz').relative_to(RUN)) if SAVE_ALL_PREDICTIONS else '' for i in range(len(table))]
        frames.append(table)
        summary = meter.compute()
        summaries.append({'model': name, 'condition': condition, **summary})
        table.to_csv(destination / 'per_image.csv', index=False)
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
results = pd.concat(frames, ignore_index=True)
results['predicted_label'] = (results.image_probability >= IMAGE_THRESHOLD).astype(int)
results.to_csv(RUN / 'per_image_results.csv', index=False)
metadata = {'dataset_root': str(DATA_ROOT), 'manifest_sha256': digest(MANIFEST),
    'stats_sha256': digest(STATS_PATH), 'audit_sha256': digest(AUDIT_PATH),
    'models': models_metadata, 'seed': SEED, 'device': str(DEVICE), 'precision': 'float32',
    'image_threshold': IMAGE_THRESHOLD, 'pixel_threshold': PIXEL_THRESHOLD,
    'mask_policy': 'image_gated: predicted image label 0 forces an empty final mask',
    'mask_foreground': MASK_FOREGROUND, 'conditions': CONDITIONS,
    'python': platform.python_version(),
    'versions': {p: importlib.metadata.version(p) for p in ['torch','torchvision','timm','numpy','scipy','pillow','pandas','matplotlib','scikit-learn']}}
(RUN / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(RUN / 'summary.json').write_text(json.dumps(summaries, indent=2, allow_nan=False), encoding='utf-8')
score_rows = []
for s in summaries:
    score_rows.append({'model': s['model'], 'condition': s['condition'],
        **{'image_'+k:v for k,v in s['detection'].items()},
        **{'pixel_'+k:v for k,v in s['localization'].items()}})
scores = pd.DataFrame(score_rows)
scores.to_csv(RUN / 'summary.csv', index=False)
display(scores)


## 5. Image detection: confusion matrix, ROC, PR, calibration and threshold sensitivity

Confusion matrices use true labels as rows and predicted labels as columns. ROC compares true-positive
rate with false-positive rate across thresholds. Average precision summarizes the precision-recall curve;
its no-skill reference is the forged-image prevalence. Reliability curves compare mean predicted probability
with observed forged frequency; Brier score is the mean squared probability error (lower is better).
ECE uses ten equal-width bins and is descriptive, not proof of calibration. ROC/PR ranking summaries require
both image classes; missing classes are annotated. Scores are affected by the dataset's class balance.


In [ ]:
clean = results[results.condition.eq('clean')].copy()
calibration_rows = []
for n, (name, t) in enumerate(clean.groupby('model', sort=False)):
    y, p = t.label.to_numpy(), t.image_probability.to_numpy()
    fig, ax = plt.subplots(2, 3, figsize=(16, 9), layout='constrained')
    cm = confusion_matrix(y, p >= IMAGE_THRESHOLD, labels=[0,1])
    ax[0,0].imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2): ax[0,0].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=24)
    ax[0,0].set(xticks=[0,1], yticks=[0,1], xticklabels=['Authentic','Forged'],
        yticklabels=['Authentic','Forged'], xlabel='Predicted class', ylabel='True class', title='Image confusion matrix')
    if len(np.unique(y)) == 2:
        fpr,tpr,_ = roc_curve(y,p)
        ax[0,1].plot(fpr,tpr,color=COLORS[0],label=f'ROC-AUC = {roc_auc_score(y,p):.3f}')
        ax[0,1].plot([0,1],[0,1],'--',color='gray'); ax[0,1].legend()
        precision,recall,_ = precision_recall_curve(y,p)
        ax[0,2].plot(recall,precision,color=COLORS[1],label=f'AP = {average_precision_score(y,p):.3f}')
        ax[0,2].axhline(y.mean(),ls='--',color='gray',label=f'Prevalence = {y.mean():.2f}'); ax[0,2].legend()
    else:
        for a in ax[0,1:]: a.text(.5,.5,'Requires both image classes',ha='center',transform=a.transAxes)
    ax[0,1].set(title='Image ROC',xlabel='False-positive rate',ylabel='True-positive rate',xlim=(0,1),ylim=(0,1.02))
    ax[0,2].set(title='Image precision-recall',xlabel='Recall',ylabel='Precision',xlim=(0,1),ylim=(0,1.02))
    bins = np.minimum((p*10).astype(int),9)
    cal = pd.DataFrame({'bin':bins,'p':p,'y':y}).groupby('bin').agg(confidence=('p','mean'),frequency=('y','mean'),count=('y','size'))
    ece = float((abs(cal.confidence-cal.frequency)*cal['count']).sum()/len(y))
    calibration_rows.append({'model':name,'brier':brier_score_loss(y,p),'ece_10_bins':ece})
    ax[1,0].plot([0,1],[0,1],'--',color='gray')
    ax[1,0].plot(cal.confidence,cal.frequency,'o-',color=COLORS[2], label=f'Brier={brier_score_loss(y,p):.3f}; ECE={ece:.3f}')
    ax[1,0].legend(fontsize=10); ax[1,0].set(title='Probability calibration',xlabel='Mean P(forged)',ylabel='Observed forged frequency',xlim=(0,1),ylim=(0,1))
    for label,color in zip([0,1],COLORS):
        ax[1,1].hist(p[y==label],bins=np.linspace(0,1,21),alpha=.6,label=['Authentic','Forged'][label],color=color)
    ax[1,1].axvline(IMAGE_THRESHOLD,color='black',ls='--'); ax[1,1].legend()
    ax[1,1].set(title='Image score distributions',xlabel='P(forged)',ylabel='Images')
    threshold_rows = pd.DataFrame([detection_metrics(y,p,x) for x in np.linspace(0,1,101)])
    threshold_rows.to_csv(RUN / f'image_threshold_sensitivity_{n}.csv',index=False)
    for metric in ['f1','precision','recall']:
        ax[1,2].plot(threshold_rows.threshold,threshold_rows[metric],label=metric.capitalize())
    ax[1,2].axvline(IMAGE_THRESHOLD,color='black',ls='--',label='Frozen operating point'); ax[1,2].legend(fontsize=10)
    ax[1,2].set(title='Descriptive threshold sensitivity',xlabel='Image threshold',ylabel='Score',ylim=(0,1.02))
    fig.suptitle(f'{name} | IMD2020 clean test', fontsize=20)
    export(fig, f'04_detection_dashboard_{n}', f'{name}: image-level detection and calibration. Threshold sensitivity is exploratory; the configured operating point remains frozen.')
pd.DataFrame(calibration_rows).to_csv(RUN / 'calibration.csv',index=False)


## 6. Localization, region size, source-group uncertainty and model comparison

Dice = 2TP/(2TP+FP+FN), IoU = TP/(TP+FP+FN). Both quantify foreground overlap;
Dice and pixel F1 are identical here and should not be presented as independent evidence.
Macro averaging gives equal weight to forged images. Micro overlap pools forged-image pixel counts and favors
large regions/images. Bootstrap intervals resample **source folders**, keeping correlated edits and originals
together; 95% percentile intervals quantify sampling variability conditional on this dataset and fixed model.
They do not measure training-seed uncertainty or correct for domain shift.


In [ ]:
ci_rows, micro_rows = [], []
for n, (name, t) in enumerate(clean.groupby('model',sort=False)):
    forged, authentic = t[t.label.eq(1)].copy(), t[t.label.eq(0)]
    fig, ax = plt.subplots(1,3,figsize=(16,4.7),layout='constrained')
    if len(forged):
        ax[0].boxplot([forged.iou.dropna(),forged.dice.dropna()],showmeans=True)
        ax[0].set_xticks([1,2], ['IoU','Dice'])
        ax[1].scatter(100*forged.foreground_fraction,forged.dice,alpha=.5,color=COLORS[0],s=18)
    ax[0].set(title='Forged-image overlap',ylabel='Score',ylim=(0,1.02))
    ax[1].set(title='Performance vs region size',xlabel='Ground-truth forged area (%)',ylabel='Dice',ylim=(0,1.02))
    ax[2].hist(100*authentic.predicted_fraction,bins=25,color=COLORS[1])
    ax[2].set(title='Authentic false-positive area',xlabel='Pixels predicted forged (%)',ylabel='Authentic images')
    fig.suptitle(name)
    export(fig,f'05_localization_{n}', 'Localization on forged images and spurious mask area on authentic images. Small regions may be harder to localize.')
    tp,fp,fn = forged[['tp','fp','fn']].sum()
    micro_rows.append({'model':name,'forged_micro_iou':tp/(tp+fp+fn) if tp+fp+fn else None,
        'forged_micro_dice':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None})
    # Pre-aggregate additive statistics by source group; bootstrap cost is independent of total pixels.
    t = t.copy()
    t['correct'] = t.label.eq(t.predicted_label).astype(int)
    grouped = t.groupby('group').agg(n=('label','size'),correct=('correct','sum'),
        dice_sum=('dice','sum'),dice_n=('dice','count'),iou_sum=('iou','sum'),iou_n=('iou','count'))
    values = grouped.to_numpy(float)
    rng = np.random.default_rng(SEED)
    distributions = {'image_accuracy':[], 'macro_dice':[], 'macro_iou':[]}
    for _ in range(BOOTSTRAPS):
        total = values[rng.integers(0,len(values),len(values))].sum(axis=0)
        distributions['image_accuracy'].append(total[1]/total[0])
        distributions['macro_dice'].append(total[2]/total[3] if total[3] else np.nan)
        distributions['macro_iou'].append(total[4]/total[5] if total[5] else np.nan)
    for metric, vals in distributions.items():
        vals = np.asarray(vals); valid = vals[np.isfinite(vals)]
        point = t.correct.mean() if metric=='image_accuracy' else t[metric.replace('macro_','')].mean()
        low,high = np.quantile(valid,[.025,.975]) if valid.size else [np.nan,np.nan]
        ci_rows.append({'model':name,'metric':metric,'estimate':point,'ci_low':low,'ci_high':high,
            'source_groups':len(values),'valid_bootstrap_replicates':len(valid)})
ci = pd.DataFrame(ci_rows)
ci.to_csv(RUN / 'group_bootstrap_ci.csv',index=False)
pd.DataFrame(micro_rows).to_csv(RUN / 'micro_localization.csv',index=False)
display(ci)
if audit.group.nunique() < 20:
    print('Caution: very few source groups; bootstrap intervals are unstable, especially for the preview manifest.')
fig, ax = plt.subplots(figsize=(12,5),layout='constrained')
for j, metric in enumerate(['image_accuracy','macro_dice','macro_iou']):
    subset = ci[ci.metric.eq(metric)]
    x = np.arange(len(subset))+(j-1)*.22
    ax.bar(x,subset.estimate,width=.22,color=COLORS[j],label=metric.replace('_',' ').title())
    # Draw percentile bounds directly: a point estimate need not lie inside its percentile interval.
    ax.vlines(x,subset.ci_low,subset.ci_high,color='black',lw=1.4)
    ax.plot(x,subset.ci_low,'_',color='black'); ax.plot(x,subset.ci_high,'_',color='black')
ax.set(xticks=np.arange(clean.model.nunique()),xticklabels=list(clean.model.unique()),
    ylabel='Score (95% source-group bootstrap interval)',ylim=(0,1.05),title='Clean external-test performance')
ax.legend()
export(fig,'06_model_comparison_ci','Frozen-threshold test metrics with source-group bootstrap intervals. Overlapping intervals are not a paired significance test.')


## 7. Original image, pixel ground truth, predicted mask, labels and error maps

Each example includes the source original, actual input, binary GT, predicted binary mask, probability map,
and pixel errors. **Green = correctly found forgery (TP), orange = spurious forgery (FP), magenta = missed
forgery (FN), dark = correct background (TN).** Image labels and predicted image labels are printed separately.
Examples are explicitly selected as best, median, worst forged-image Dice and largest authentic false-positive
area, so they illustrate both successes and failure modes. They are not a random performance estimate.


In [ ]:
example_rows = []
if not SAVE_ALL_PREDICTIONS:
    print('Set SAVE_ALL_PREDICTIONS=True and rerun evaluation to produce mask galleries and pixel-threshold plots.')
else:
    for n, (name,t) in enumerate(clean.groupby('model',sort=False)):
        forged = t[t.label.eq(1)].sort_values(['dice','sample_id'])
        categories = {'worst':forged.head(EXAMPLES_PER_CATEGORY),
            'best':forged.tail(EXAMPLES_PER_CATEGORY),
            'median':forged.iloc[max(0,len(forged)//2-EXAMPLES_PER_CATEGORY//2):len(forged)//2+(EXAMPLES_PER_CATEGORY+1)//2],
            'authentic_fp':t[t.label.eq(0)].nlargest(EXAMPLES_PER_CATEGORY,'predicted_fraction')}
        for category, selected in categories.items():
            for k, (_, row) in enumerate(selected.iterrows()):
                idx = int(manifest.index[manifest.id.eq(row.sample_id)][0])
                raw = dataset[idx]
                with np.load(RUN / row.prediction_file) as saved: p = saved['probability']
                gt = raw['mask'][0].astype(bool)
                pred = (p >= PIXEL_THRESHOLD) & (row.image_probability >= IMAGE_THRESHOLD)
                rgb = np.moveaxis(raw['rgb'],0,-1)
                errors = np.zeros((*gt.shape,3),dtype=np.uint8)+35
                errors[gt & pred] = [0,158,115]
                errors[~gt & pred] = [230,159,0]
                errors[gt & ~pred] = [204,121,167]
                source_row = manifest[manifest.group.eq(row.group) & manifest.label.eq(0)]
                fig, axes = plt.subplots(2,3,figsize=(16,9),layout='constrained')
                if len(source_row):
                    with Image.open(DATA_ROOT / source_row.iloc[0].image) as im: axes[0,0].imshow(im.convert('RGB'))
                axes[0,0].set_title('Source original | true label 0')
                axes[0,1].imshow(rgb); axes[0,1].set_title(f'Actual input | true label {int(row.label)}')
                axes[0,2].imshow(gt,cmap='gray',vmin=0,vmax=1,interpolation='nearest'); axes[0,2].set_title('GT pixel mask | white = forged')
                axes[1,0].imshow(pred,cmap='gray',vmin=0,vmax=1,interpolation='nearest'); axes[1,0].set_title(f'Predicted mask | pixel threshold {PIXEL_THRESHOLD:g}')
                axes[1,1].imshow(rgb)
                heat = axes[1,1].imshow(p,cmap='magma',vmin=0,vmax=1,alpha=.65)
                axes[1,1].set_title('Final probability overlay | image-gated'); fig.colorbar(heat,ax=axes[1,1],shrink=.7)
                axes[1,2].imshow(errors,interpolation='nearest'); axes[1,2].set_title('Pixel errors: TP / FP / FN')
                axes[1,2].legend(handles=[Patch(color=c,label=l) for c,l in [('#009E73','TP'),('#E69F00','FP'),('#CC79A7','FN'),('#232323','TN')]],loc='upper center',ncol=4,fontsize=9)
                for a in axes.flat: a.axis('off')
                dice_text = 'N/A (authentic)' if pd.isna(row.dice) else f'{row.dice:.3f}'
                fig.suptitle(f'{name} | {category} | {row.sample_id}\nTrue image label={row.label}; predicted label={row.predicted_label}; P(forged)={row.image_probability:.3f}; Dice={dice_text}',fontsize=16)
                figure_name = f'07_example_model{n}_{category}_{k}'
                export(fig,figure_name,f'{category} example: source original, model input, aligned GT and predicted pixel masks, probabilities, and pixel errors. Source original is contextual only.')
                example_rows.append({'figure':figure_name,'id':row.sample_id,'model':name,'category':category})
    pd.DataFrame(example_rows).to_csv(RUN / 'example_selection.csv',index=False)


## 8. Pixel-threshold sensitivity and robustness

Pixel-threshold curves average foreground Dice/IoU over forged images and report authentic false-positive
area separately. They do not change the configured threshold. Robustness compares each degradation with
the same model's clean score using the same test images. A downscale condition downsamples then restores
RGB to the original grid; GT masks are untouched. DCT and SRM are recomputed from degraded RGB.


In [ ]:
if SAVE_ALL_PREDICTIONS:
    grid = np.linspace(0,1,21)
    threshold_tables = []
    for n,(name,t) in enumerate(clean.groupby('model',sort=False)):
        dice_sums=np.zeros(len(grid)); iou_sums=np.zeros(len(grid)); area_sums=np.zeros(len(grid))
        nf=na=0
        for _,row in tqdm(t.iterrows(),total=len(t),desc='Pixel threshold sensitivity'):
            idx = int(manifest.index[manifest.id.eq(row.sample_id)][0])
            gt = dataset[idx]['mask'][0].astype(bool)
            with np.load(RUN / row.prediction_file) as saved: p=saved['probability']
            for j,threshold in enumerate(grid):
                pred = (p >= threshold) & (row.image_probability >= IMAGE_THRESHOLD)
                if row.label==1:
                    tp=np.count_nonzero(gt & pred); fp=np.count_nonzero(~gt & pred); fn=np.count_nonzero(gt & ~pred)
                    dice_sums[j]+=2*tp/max(2*tp+fp+fn,1); iou_sums[j]+=tp/max(tp+fp+fn,1)
                else: area_sums[j]+=pred.mean()
            nf+=int(row.label==1); na+=int(row.label==0)
        th=pd.DataFrame({'threshold':grid,'macro_dice':dice_sums/nf if nf else np.nan,
            'macro_iou':iou_sums/nf if nf else np.nan,'authentic_fp_area':area_sums/na if na else np.nan,'model':name})
        threshold_tables.append(th)
        fig,ax=plt.subplots(figsize=(10,5),layout='constrained')
        for metric in ['macro_dice','macro_iou','authentic_fp_area']: ax.plot(th.threshold,th[metric],label=metric.replace('_',' '))
        ax.axvline(PIXEL_THRESHOLD,color='black',ls='--',label='Frozen operating point')
        ax.set(title=f'{name}: pixel-threshold sensitivity',xlabel='Pixel probability threshold',ylabel='Score / fraction',ylim=(0,1.02)); ax.legend()
        export(fig,f'08_pixel_threshold_{n}','Exploratory pixel-threshold sensitivity; threshold is not selected using this test set.')
    pd.concat(threshold_tables).to_csv(RUN / 'pixel_threshold_sensitivity.csv',index=False)

    # Stream score histograms instead of concatenating billions of pixel probabilities.
    # These pooled pixel curves are binned approximations, distinct from exact macro image-wise AUC above.
    for n,(name,t) in enumerate(clean.groupby('model',sort=False)):
        edges=np.linspace(0,1,4097)
        positive=np.zeros(4096,dtype=np.int64); negative=positive.copy()
        for _,row in tqdm(t.iterrows(),total=len(t),desc='Pooled pixel ROC / PR'):
            idx=int(manifest.index[manifest.id.eq(row.sample_id)][0])
            gt=dataset[idx]['mask'][0].astype(bool)
            with np.load(RUN / row.prediction_file) as saved: p=saved['probability']
            positive+=np.histogram(p[gt],bins=edges)[0]
            negative+=np.histogram(p[~gt],bins=edges)[0]
        fig,ax=plt.subplots(1,2,figsize=(13,5),layout='constrained')
        if positive.sum() and negative.sum():
            tp=np.r_[0,np.cumsum(positive[::-1])].astype(float)
            fp=np.r_[0,np.cumsum(negative[::-1])].astype(float)
            recall=tp/positive.sum(); fpr=fp/negative.sum()
            precision=np.divide(tp,tp+fp,out=np.ones_like(tp),where=(tp+fp)>0)
            auc=float(np.sum(np.diff(fpr)*(recall[1:]+recall[:-1])/2))
            ap=float(np.sum(np.diff(recall)*precision[1:]))
            pd.DataFrame({'fpr':fpr,'recall':recall,'precision':precision}).to_csv(RUN / f'pooled_pixel_curves_{n}.csv',index=False)
            ax[0].plot(fpr,recall,label=f'Binned pooled AUC = {auc:.3f}',color=COLORS[0])
            ax[0].plot([0,1],[0,1],'--',color='gray'); ax[0].legend()
            ax[1].plot(recall,precision,label=f'Binned pooled AP = {ap:.3f}',color=COLORS[1])
            prevalence=positive.sum()/(positive.sum()+negative.sum())
            ax[1].axhline(prevalence,ls='--',color='gray',label=f'Pixel prevalence = {prevalence:.3f}'); ax[1].legend()
        else:
            for a in ax: a.text(.5,.5,'Requires foreground and background pixels',ha='center',transform=a.transAxes)
        ax[0].set(title='Pooled pixel ROC',xlabel='Pixel false-positive rate',ylabel='Pixel true-positive rate',xlim=(0,1),ylim=(0,1.02))
        ax[1].set(title='Pooled pixel precision-recall',xlabel='Pixel recall',ylabel='Pixel precision',xlim=(0,1),ylim=(0,1.02))
        fig.suptitle(name)
        export(fig,f'08b_pooled_pixel_curves_{n}','Approximate pooled pixel curves using 4096 probability bins, including authentic images. Large images contribute more pixels; these differ from exact macro pixel AUC.')

if len(CONDITIONS)>1:
    fig,axes=plt.subplots(1,3,figsize=(16,5),layout='constrained')
    for name,t in scores.groupby('model',sort=False):
        t=t.set_index('condition').reindex(CONDITIONS)
        for ax,metric,title in zip(axes,['image_roc_auc','pixel_macro_dice','pixel_pristine_false_positive_area'],
            ['Image ROC-AUC (higher better)','Macro pixel Dice (higher better)','Authentic FP area (lower better)']):
            ax.plot(CONDITIONS,t[metric],marker='o',label=name); ax.set(title=title,ylabel='Score / fraction',ylim=(0,1.02)); ax.tick_params(axis='x',rotation=30)
    axes[0].legend(fontsize=10)
    export(fig,'09_robustness','Paired test-image performance under configured degradations. Each condition is independently generated from original RGB.')
    delta=scores.copy()
    for metric in ['image_roc_auc','pixel_macro_dice','pixel_pristine_false_positive_area']:
        base=scores[scores.condition.eq('clean')].set_index('model')[metric]
        delta[metric+'_change_from_clean']=delta[metric]-delta.model.map(base)
    delta.to_csv(RUN / 'robustness_change_from_clean.csv',index=False)
else:
    print('Robustness disabled. Configure additional CONDITIONS before running evaluation to create this comparison.')


## 9. Failure tables, measured conclusions and PPT export guide

Read image detection and mask localization together: a correct forged-image decision does not guarantee
correct localization. Compare class balance, foreground size, false-positive area, ROC-AUC and overlap.
High ROC-AUC with weak frozen-threshold F1 can indicate an operating-point/calibration issue; it does not
justify tuning on test data. Low clean scores may reflect domain shift, a preprocessing mismatch, or mask
annotation issues; these plots alone cannot identify the cause.

For slides use the exported 300-DPI PNG files. Only PNG figures are generated.
Keep aspect ratios and use light slide backgrounds,
and cite the test manifest, model stage, thresholds, sample count and source-group count in your results slide.
Suggested flow: architecture → dataset → detection → localization/CI → qualitative successes/failures → robustness → limitations.


In [ ]:
failures=clean[clean.label.ne(clean.predicted_label)].copy()
failures['confidence_in_wrong_class']=np.where(failures.predicted_label.eq(1),failures.image_probability,1-failures.image_probability)
failures.sort_values('confidence_in_wrong_class',ascending=False).to_csv(RUN / 'image_detection_failures.csv',index=False)
clean[clean.label.eq(1)].sort_values('dice').to_csv(RUN / 'localization_ranked_worst_first.csv',index=False)
size_analysis=clean[clean.label.eq(1)].copy()
size_analysis['region_size']=pd.cut(100*size_analysis.foreground_fraction,[0,1,5,20,100],
    labels=['0-1%','1-5%','5-20%','20-100%'],include_lowest=True)
display(size_analysis.groupby(['model','region_size'],observed=True).agg(images=('sample_id','size'),dice=('dice','mean'),iou=('iou','mean')))
size_analysis.groupby(['model','region_size'],observed=True).agg(images=('sample_id','size'),dice=('dice','mean'),iou=('iou','mean')).to_csv(RUN / 'region_size_analysis.csv')

lines=['# IMD2020 measured evaluation report', '',
    f'Test: {len(dataset)} images, {manifest.group.nunique()} source groups. Manifest: {MANIFEST.name}.',
    f'Frozen thresholds: image={IMAGE_THRESHOLD}, pixel={PIXEL_THRESHOLD}.',
    'Pixel overlap averages include forged images only. No thresholds were fitted on test data.', '']
def fmt(value): return 'N/A' if pd.isna(value) else f'{value:.3f}'
for name,t in clean.groupby('model',sort=False):
    s=scores[scores.model.eq(name)&scores.condition.eq('clean')].iloc[0]
    lines.extend([f'## {name}',
        f'Image accuracy {fmt(s.image_accuracy)}, F1 {fmt(s.image_f1)}, ROC-AUC {fmt(s.image_roc_auc)}.',
        f'Forged-image macro Dice {fmt(s.pixel_macro_dice)}, macro IoU {fmt(s.pixel_macro_iou)}; authentic false-positive area {fmt(s.pixel_pristine_false_positive_area)}.',
        f'Incorrect image decisions: {int(t.label.ne(t.predicted_label).sum())}/{len(t)}.',
        f'Median preprocessing + inference + metric time: {t.seconds.median():.3f} s/image (hardware-specific; includes first-pass overhead).', ''])
lines.extend(['## Interpretation limits',
    'Confirm that this external dataset and related source images were not used in training or selection.',
    'Inspect mask polarity/alignment and both image classes. Source-related samples are correlated.',
    'Bootstrap intervals describe source-group sampling uncertainty, not training uncertainty.',
    'Qualitative galleries use declared best/median/worst selection; use aggregate scores for claims.',
    'Preview runs are workflow checks, not full-dataset results. Test threshold curves are descriptive only.'])
(RUN / 'analysis_report.md').write_text('\n'.join(lines),encoding='utf-8')
pd.DataFrame(figure_index).to_csv(RUN / 'figure_captions.csv',index=False)
display(pd.DataFrame(figure_index))
print('All outputs:',RUN)
print('PPT images:',RUN / 'figures')
print('Detailed measured report:',RUN / 'analysis_report.md')
